# Lesson 1: Building a "Manual Reader" with RAG

Welcome to Part 1!

In this notebook, we are going to fix **The Knowledge Gap** by teaching our AI to read *your* private technical manuals before it answers. We will do this using a technique called **Retrieval-Augmented Generation (RAG)**.

### Concept: The Knowledge Gap and RAG

Large Language Models are trained on a massive snapshot of the public internet. If a technical manual was written after its training cutoff date, or if it's a private company document, the LLM simply hasn't read it. If you ask it a question about that document, it will either say "I don't know" or worse, it will hallucinate (make something up).

**The Solution: Retrieval-Augmented Generation (RAG)**
Instead of retraining the entire model (which is wildly expensive), we give the LLM an open-book test.

Here is the architecture we will build today:
1. **Ingestion (PDF):** We load your technical manual.
2. **Chunking:** We split the manual into smaller, bite-sized paragraphs.
3. **Embeddings:** We convert these text chunks into numerical vectors (lists of numbers) that capture their meaning.
4. **Vector Database (FAISS):** We store these vectors in a specialized database.
5. **Retrieval:** When a user asks a question, we convert the question into a vector, search the database for the top 3 most similar chunks, and retrieve them.
6. **Generation (LLM):** We feed those specific chunks to Llama 3.2 (running locally via Ollama) along with the question and say, *"Based ONLY on these chunks, answer the question."*

### Step 1: Setting up our RAG Tools and Ollama

To build our pipeline, we need a few tools. We will use **LangChain** to orchestrate, and **Ollama** to run models locally.

| Library | Purpose |
| :--- | :--- |
| **`langchain` & `langchain-ollama`** | The orchestration framework and Ollama integration. |
| **`pypdf`** | A tool to extract text from PDF files. |
| **`faiss-cpu`** | A high-performance Vector Database created by Meta to store and search our embeddings locally. |

In [2]:
# Run this cell to install the necessary libraries
!pip install -q langchain langchain-community langchain-ollama pypdf faiss-cpu langchain-text-splitters

In [5]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
import os
# NOTE: Edit the path below to match the exact location where you 
# saved the course materials in your personal Google Drive workspace.
# For example: %cd /content/drive/MyDrive/Your_Folder_Name/

%cd /content/drive/MyDrive/KE_Course/
print("Current Directory:", os.getcwd())

/content/drive/MyDrive/KE_Course
Current Directory: /content/drive/MyDrive/KE_Course


### Step 2: Starting the Ollama Server & Pulling Models

Now we set up Ollama in the background of our Colab environment and download the models we need:
- **llama3.2**: For generating text.
- **nomic-embed-text**: For generating vector embeddings.

In [4]:
import os
import threading
import subprocess
import time

# 1. Install Ollama
!sudo apt-get update -y > /dev/null 2>&1
!sudo apt-get install -y pciutils zstd > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh

# 2. Start the Ollama server in the background
def run_ollama_serve():
    env = os.environ.copy()
    subprocess.Popen(["ollama", "serve"], env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

threading.Thread(target=run_ollama_serve, daemon=True).start()
time.sleep(5) # Give the server a few seconds to boot up

# 3. Pull the models
!ollama pull llama3.2
!ollama pull nomic-embed-text

print("\nOllama is ready!")

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.



Ollama is ready!


### Step 3: Loading the PDF and Splitting into Chunks

We can't feed a 100-page manual into an LLM all at once; it will forget things and run out of memory. We need to cut the document into overlapping chunks.

*Before running the next cell, make sure you have a PDF file named `sample_manual.pdf` in your directory!*

In [8]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Load the PDF
loader = PyPDFLoader("sample_manual.pdf")
documents = loader.load()
print(f"Loaded {len(documents)} pages from the PDF.")

# 2. Split the text into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
chunks = text_splitter.split_documents(documents)
print(f"Split the document into {len(chunks)} chunks.")

Loaded 2 pages from the PDF.
Split the document into 4 chunks.


### Step 4: Embeddings and the FAISS Vector Database

Now we translate our text chunks into vectors using Ollama's `nomic-embed-text` model. Then, we load these vectors into **FAISS**, which acts as our search engine.

In [9]:
from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import FAISS

# 1. Load the embedding model
print("Loading Ollama embedding model...")
embeddings = OllamaEmbeddings(model="nomic-embed-text")

# 2. Create the Vector Database
print("Creating FAISS Vector Database. This might take a moment...")
vector_db = FAISS.from_documents(chunks, embeddings)

# 3. Create a retriever object that fetches the top 3 most relevant chunks
retriever = vector_db.as_retriever(search_kwargs={"k": 3})
print("Vector Database is ready!")

Loading Ollama embedding model...
Creating FAISS Vector Database. This might take a moment...
Vector Database is ready!


### Step 5: Building the "Manual Reader" Prompt

Now we tie it all together. When a user asks a question, we will:
1. Search FAISS for the top 3 chunks.
2. Inject those chunks into a strict prompt.
3. Send the prompt to Llama 3.2 via Ollama.

In [10]:
from langchain_ollama import ChatOllama

# The user's question
question = "What is the recommended operating temperature for the main sensor?"

# 1. Retrieve the relevant chunks from the database
retrieved_docs = retriever.invoke(question)

# Combine the retrieved text into a single string
context = "\n\n".join([doc.page_content for doc in retrieved_docs])

# 2. Format the prompt
prompt = f"""You are an expert technical assistant. Answer the user's question based ONLY on the context provided below.
If the answer is not contained in the context, say "I cannot find the answer in the provided manual." Do not use outside knowledge.

Context:
{context}

Question: {question}
"""

print("Generating answer...\n")

# 3. Generate the response using Ollama
llm = ChatOllama(model="llama3.2", temperature=0.1)
response = llm.invoke(prompt)

# Print the answer
print(f"Question: {question}\n")
print(f"Answer: {response.content}")

Generating answer...

Question: What is the recommended operating temperature for the main sensor?

Answer: The recommended operating temperature for the main sensor is strictly between 18°C and 24°C (64°F - 75°F).
